# 03 - Build Nominatim Server (Import OSM Data)

Full Nominatim import pipeline that mirrors `scripts/03_build_nominatim_server.sh`.

Key differences from the local scripts:
- **No `.env` file** - authenticates via `WorkspaceClient` running in the same workspace
- **Python threading** for background token refresh (instead of bash background process)
- **pgpass file** approach for CLI tools (`nominatim`, `osm2pgsql`, `psql`)
- **Automatic SSL-disconnect retry** with `--continue indexing`

## Prerequisites
1. Run `01_setup_postgis` (PostGIS + hstore extensions installed)
2. Run `02_download_osm_data` (OSM `.pbf` files on driver disk)
3. `nominatim` CLI available on the cluster (install via init script or pip)

## Configuration

In [ ]:
%pip install databricks-sdk==0.99.0 nominatim-db
%restart_python

In [ ]:
dbutils.widgets.text("pg_project_id", "nominatim-lakebase", "Lakebase Project ID")
dbutils.widgets.text("pg_branch_id", "production", "Lakebase Branch ID")
dbutils.widgets.text("pg_user", "justin.monaldo@databricks.com", "PG User")
dbutils.widgets.text("pg_database", "nominatim", "PG Database")
dbutils.widgets.text("pg_port", "5432", "PG Port")
dbutils.widgets.text("osm_data_dir", "/Volumes/justinm/nominatim/osm_data", "OSM data directory")
dbutils.widgets.text("regions", "monaco", "Regions (comma-separated, must match downloaded files)")
dbutils.widgets.text("import_threads", "0", "Import threads (0 = auto-detect)")
dbutils.widgets.text("cache_mb", "0", "osm2pgsql cache MB (0 = auto-detect)")
dbutils.widgets.text("token_refresh_seconds", "2700", "Token refresh interval (seconds)")
dbutils.widgets.text("import_retry_max", "2", "Max retries on SSL disconnect")

In [ ]:
%run ./_helpers

In [ ]:
%run ./OSM_SOURCES

In [ ]:
import os
import re
import time
import signal
import threading
import subprocess
from pathlib import Path
from datetime import datetime

# Read widgets
pg_project_id = dbutils.widgets.get("pg_project_id")
pg_branch_id = dbutils.widgets.get("pg_branch_id")
pg_user = dbutils.widgets.get("pg_user")
pg_database = dbutils.widgets.get("pg_database")
pg_port = dbutils.widgets.get("pg_port")
osm_data_dir = Path(dbutils.widgets.get("osm_data_dir"))
regions_raw = dbutils.widgets.get("regions").strip()
import_threads = int(dbutils.widgets.get("import_threads"))
cache_mb = int(dbutils.widgets.get("cache_mb"))
TOKEN_REFRESH_SECONDS = int(dbutils.widgets.get("token_refresh_seconds"))
IMPORT_RETRY_MAX = int(dbutils.widgets.get("import_retry_max"))

PGCONNECT_TIMEOUT = 15
PROJECT_DIR = Path("/tmp/nominatim_project")

## Resolve OSM Files

In [ ]:
regions = [r.strip() for r in regions_raw.split(",") if r.strip()]
if not regions:
    raise ValueError("No regions specified")

osm_files = []
for r in regions:
    if r not in OSM_SOURCES:
        raise ValueError(f"Unknown region '{r}'")
    filename = OSM_SOURCES[r].split("/")[-1]
    path = osm_data_dir / filename
    if not path.exists():
        raise FileNotFoundError(
            f"OSM file not found: {path}\n"
            f"Run 02_download_osm_data first with region '{r}'"
        )
    osm_files.append(path)

# De-duplicate (e.g. multiple GCC states map to the same file)
osm_files = list(dict.fromkeys(osm_files))

print(f"OSM files to import ({len(osm_files)}):")
for f in osm_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f}  ({size_mb:.1f} MB)")

## Auto-detect Threads and Cache

In [ ]:
def detect_threads():
    """Auto-detect a reasonable thread count."""
    try:
        cpu_count = os.cpu_count() or 4
    except Exception:
        cpu_count = 4
    return max(2, min(cpu_count, 12))


def detect_cache():
    """Auto-detect osm2pgsql cache size (roughly 1/3 of total memory)."""
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    total_kb = int(line.split()[1])
                    total_mb = total_kb // 1024
                    cache = total_mb // 3
                    return max(2000, min(cache, 16000))
    except Exception:
        pass
    return 2000


if import_threads <= 0:
    import_threads = detect_threads()
if cache_mb <= 0:
    cache_mb = detect_cache()

print(f"Import threads: {import_threads}")
print(f"osm2pgsql cache: {cache_mb} MB")

## Authenticate and Setup pgpass

In [ ]:
pg_env = get_pg_env(
    project_id=pg_project_id,
    branch_id=pg_branch_id,
    user=pg_user,
    database=pg_database,
    port=pg_port,
)
pg_host = pg_env["PGHOST"]

pgpass_path = write_pgpass_file(pg_env)

# Build the shell env: inherit system PATH etc., add PG vars, use PGPASSFILE
shell_env = {**os.environ, **pg_env}
shell_env["PGPASSFILE"] = pgpass_path
shell_env["PGCONNECT_TIMEOUT"] = str(PGCONNECT_TIMEOUT)
# Remove PGPASSWORD so CLI tools use PGPASSFILE instead
shell_env.pop("PGPASSWORD", None)

print(f"pgpass file: {pgpass_path}")
print(f"Database: {pg_user}@{pg_host}:{pg_port}/{pg_database}")
print(f"Token refresh interval: {TOKEN_REFRESH_SECONDS}s")

## Background Token Refresher

Runs a daemon thread that refreshes the Lakebase OAuth token in the pgpass
file every N seconds (default 45 min). Tokens expire after 1 hour.

In [ ]:
refresher_stop = threading.Event()


def token_refresher_loop():
    """Background thread: refresh pgpass token periodically."""
    while not refresher_stop.wait(timeout=TOKEN_REFRESH_SECONDS):
        try:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"{ts} [Background] Refreshing OAuth token...")
            new_token = refresh_pgpass(pgpass_path, pg_env)
            print(f"{ts} [Background] Token refreshed in pgpass (len={len(new_token)})")
        except Exception as e:
            ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            print(f"{ts} [Background] WARNING: token refresh failed: {e}")


refresher_thread = threading.Thread(target=token_refresher_loop, daemon=True)
refresher_thread.start()
print(f"Background token refresher started (interval: {TOKEN_REFRESH_SECONDS}s)")

## Helper: run psql

In [ ]:
def run_psql(sql_cmd, database="postgres"):
    """Execute a SQL statement via psql against the Lakebase instance."""
    dsn = (
        f"host={pg_host} port={pg_port} dbname={database} "
        f"user={pg_user} sslmode=require connect_timeout={PGCONNECT_TIMEOUT}"
    )
    result = subprocess.run(
        ["psql", "-w", "-v", "ON_ERROR_STOP=1", dsn, "-tAc", sql_cmd],
        env=shell_env,
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f"psql failed: {result.stderr.strip()}")
    return result.stdout.strip()

## Setup Prerequisites (www-data user, clean database)

In [ ]:
print("=" * 60)
print("Setting up prerequisites...")
print("=" * 60)
print()

# Create www-data user if needed
print("Checking for www-data user...")
user_exists = run_psql(
    "SELECT 1 FROM pg_roles WHERE rolname='www-data' LIMIT 1;"
).strip()
if user_exists != "1":
    run_psql('CREATE USER "www-data";')
    print("  Created www-data user")
else:
    print("  www-data user exists")
print()

In [ ]:
# Drop and recreate the database for a clean import
print(f"Ensuring clean database '{pg_database}'...")
db_exists = run_psql(
    f"SELECT 1 FROM pg_database WHERE datname='{pg_database}' LIMIT 1;"
).strip()

if db_exists == "1":
    print("  Database exists - dropping before import...")
    try:
        run_psql(
            f"SELECT pg_terminate_backend(pid) FROM pg_stat_activity "
            f"WHERE datname='{pg_database}' AND pid <> pg_backend_pid();"
        )
    except Exception:
        pass  # OK if no active connections
    time.sleep(1)
    run_psql(f'DROP DATABASE "{pg_database}";')
    time.sleep(1)
    print("  Database dropped")
else:
    print("  Database does not exist yet; import will initialize it")
print()

## Apply PostgreSQL Import Optimizations

In [ ]:
IMPORT_PG_SETTINGS_APPLIED = False


def apply_postgres_import_settings():
    global IMPORT_PG_SETTINGS_APPLIED
    print("=" * 60)
    print("Applying PostgreSQL user-level IMPORT optimizations...")
    print("=" * 60)

    settings = [
        ("maintenance_work_mem", "'1GB'"),
        ("work_mem", "'128MB'"),
        ("synchronous_commit", "'off'"),
        ("random_page_cost", "1.1"),
    ]
    for param, value in settings:
        try:
            run_psql(f'ALTER USER "{pg_user}" SET {param} = {value};')
            print(f"  Set {param} = {value}")
        except Exception as e:
            print(f"  Could not set {param}: {e}")

    IMPORT_PG_SETTINGS_APPLIED = True
    print()


def restore_postgres_normal_settings():
    global IMPORT_PG_SETTINGS_APPLIED
    print()
    print("=" * 60)
    print("Restoring PostgreSQL user-level settings to NORMAL...")
    print("=" * 60)

    for param in ["maintenance_work_mem", "work_mem", "random_page_cost"]:
        try:
            run_psql(f'ALTER USER "{pg_user}" RESET {param};')
            print(f"  Reset {param}")
        except Exception as e:
            print(f"  Could not reset {param}: {e}")

    try:
        run_psql(f'ALTER USER "{pg_user}" SET synchronous_commit = \'on\';')
        print("  Set synchronous_commit = on")
    except Exception as e:
        print(f"  Could not set synchronous_commit = on: {e}")

    IMPORT_PG_SETTINGS_APPLIED = False
    print()


apply_postgres_import_settings()

## Prepare Nominatim Project Directory

In [ ]:
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Write the project .env for Nominatim CLI (it reads this)
nominatim_dsn = (
    f"pgsql:dbname={pg_database};host={pg_host};port={pg_port};"
    f"user={pg_user};sslmode=require"
)

project_env_content = (
    f"PGHOST={pg_host}\n"
    f"PGPORT={pg_port}\n"
    f"PGUSER={pg_user}\n"
    f"PGDATABASE={pg_database}\n"
    f"PGSSLMODE=require\n"
    f"NOMINATIM_DATABASE_DSN={nominatim_dsn}\n"
    f"\n"
    f"NOMINATIM_IMPORT_THREADS={import_threads}\n"
    f"NOMINATIM_TOKENIZER=icu\n"
)

(PROJECT_DIR / ".env").write_text(project_env_content)
print(f"Project directory: {PROJECT_DIR}")
print(f"Nominatim DSN: {nominatim_dsn}")
print()

## Run Nominatim Import

Mirrors the retry logic from `03_build_nominatim_server.sh`:
- Fresh import on first attempt
- On SSL disconnect failure, refresh token and retry with `--continue indexing`
- Reduces thread count on retries for connection stability

In [ ]:
IMPORT_LOG = PROJECT_DIR / "import.log"


def log(msg):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"{ts} {msg}")


def has_ssl_disconnect_error(log_path):
    """Check if the import log contains SSL disconnect errors."""
    if not log_path.exists():
        return False
    text = log_path.read_text(errors="replace")
    return bool(re.search(
        r"SSL connection has been closed unexpectedly|consuming input failed",
        text,
    ))


def run_import(continue_at, threads):
    """
    Run a single nominatim import attempt.

    Args:
        continue_at: "none" for fresh import, or a stage like "indexing".
        threads: Number of import threads.

    Returns:
        subprocess return code.
    """
    cmd = ["nominatim", "import"]

    if continue_at == "none":
        for f in osm_files:
            cmd += ["--osm-file", str(f)]
    else:
        cmd += ["--continue", continue_at]

    cmd += ["--threads", str(threads), "--osm2pgsql-cache", str(cache_mb)]

    # Log header
    with open(IMPORT_LOG, "a") as lf:
        lf.write(f"\n{'=' * 60}\n")
        lf.write(f"Import attempt ({datetime.now()})\n")
        lf.write(f"Mode: {'fresh import' if continue_at == 'none' else 'continue at ' + continue_at}\n")
        lf.write(f"Threads: {threads}\n")
        lf.write(f"{'=' * 60}\n")

    start_ts = time.time()
    log(f"Running: {' '.join(cmd)}")

    with open(IMPORT_LOG, "a") as lf:
        proc = subprocess.run(
            cmd,
            cwd=str(PROJECT_DIR),
            env=shell_env,
            stdout=lf,
            stderr=subprocess.STDOUT,
        )

    elapsed = int(time.time() - start_ts)
    log(f"Import attempt finished in {elapsed}s with exit code {proc.returncode}")
    return proc.returncode

In [ ]:
print("=" * 60)
print("Starting Nominatim import...")
print("=" * 60)
print()
print(f"OSM Files:    {[str(f) for f in osm_files]}")
print(f"Threads:      {import_threads}")
print(f"Cache (MB):   {cache_mb}")
print(f"Database:     {pg_user}@{pg_host}:{pg_port}/{pg_database}")
print(f"Retry mode:   up to {IMPORT_RETRY_MAX} retries on SSL disconnect")
print()
print("This may take a while depending on data size:")
print("  Monaco:        ~5-10 minutes")
print("  Country-level: 30 minutes to several hours")
print()

# Clear log
IMPORT_LOG.write_text("")

attempt = 1
max_attempts = IMPORT_RETRY_MAX + 1
import_exit_code = 1
continue_mode = "none"
attempt_threads = import_threads

while attempt <= max_attempts:
    log(f"Starting import attempt {attempt}/{max_attempts} "
        f"(mode: {continue_mode}, threads: {attempt_threads})")

    import_exit_code = run_import(continue_mode, attempt_threads)

    if import_exit_code == 0:
        break

    # Check for SSL disconnect - retry if possible
    if attempt < max_attempts and has_ssl_disconnect_error(IMPORT_LOG):
        log("Detected SSL connection drop during import. "
            "Refreshing token and resuming at indexing...")
        refresh_pgpass(pgpass_path, pg_env)
        continue_mode = "indexing"
        if attempt_threads > 4:
            attempt_threads = 4
            log(f"Reducing retry thread count to {attempt_threads} for connection stability.")
        attempt += 1
        time.sleep(3)
        continue

    # Non-SSL failure or out of retries
    break

## Cleanup and Results

In [ ]:
# Stop the background token refresher
refresher_stop.set()

# Restore PG settings
if IMPORT_PG_SETTINGS_APPLIED:
    try:
        restore_postgres_normal_settings()
    except Exception as e:
        print(f"WARNING: Failed to restore PG settings: {e}")

# Clean up pgpass
try:
    os.remove(pgpass_path)
except OSError:
    pass

In [ ]:
if import_exit_code != 0:
    print()
    print("=" * 60)
    print(f"IMPORT FAILED (exit code: {import_exit_code})")
    print("=" * 60)
    print()
    print("Last 40 lines from import log:")
    print()
    log_lines = IMPORT_LOG.read_text(errors="replace").splitlines()
    for line in log_lines[-40:]:
        print(line)
    raise RuntimeError(f"Nominatim import failed with exit code {import_exit_code}")
else:
    print()
    print("=" * 60)
    print("Import completed successfully!")
    print("=" * 60)
    print()
    print(f"Database: {pg_database} on {pg_host}")
    print()
    print("Next steps:")
    print(f"  1. Test: cd {PROJECT_DIR} && nominatim admin --check-database")
    print("  2. Deploy the Nominatim API app")